**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `R/rstudio/R_RUN_ALL_MODELS.Rmd`, `python_prebuilt/`. Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# V05 — Counterfactual Construction Check
Verifies that `Y(0)_hat = Y_observed - effect` introduces **zero** distortion of its own.

The test builds data where the TRUE counterfactual is known (it is the DGP, not an estimate),
then confirms the constructed counterfactual's error equals **exactly** `|beta_hat - true_effect|`
for every treated-post row — i.e. all error comes from estimation, none from the construction.

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (v20.55: the engine installs a missing ESTIMATOR package itself -- wheel first, then source -- and confirms 'N of N installed' at every run; these base libraries it needs before it can start)")


In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")

sys.path.insert(0,"../All_DIDsCodes"); sys.path.insert(0,"All_DIDsCodes")
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.44")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("V05")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
C.autotune()          # v17: measures THIS machine (cores/RAM/VRAM/disk) and sets every performance knob
C.step(1,"simulate with a KNOWN true counterfactual")
rng=np.random.default_rng(201); TRUE=2.5; rows=[]; y0={}
for u in range(60):
    tr=1 if u<30 else 0; ufe=rng.normal(0,1.5)
    for t in range(2018,2026):
        tfe=rng.normal(0,.8); base=ufe+tfe+rng.normal(0,.3)
        d=1 if (tr and t>=2022) else 0
        y0[(u,t)]=base
        rows.append({"unit":u,"time":t,"treat_core":tr,"post":int(t>=2022),"y":base+TRUE*d})
d=pd.DataFrame(rows); d["did_term"]=d.treat_core*d.post
beta,_=C.estimate_twfe_did(d,"y","did_term","unit","time","unit")
C.ok(f"estimated beta={beta:.6f} (true {TRUE})")
C.done(1, next_task="next cell")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
C.step(2,"construct the counterfactual and compare to the KNOWN truth")
d["effect_applied"]=d.did_term*beta
cf=C.construct_counterfactual(d,"y","effect_applied")
cf["true_y0"]=cf.apply(lambda r: y0[(r.unit,r.time)],axis=1)
tp=cf[(cf.treat_core==1)&(cf.post==1)]
err=(tp.Y_counterfactual-tp.true_y0).abs()
exp=abs(beta-TRUE)
print(f"mean |constructed - true| = {err.mean():.8f}")
print(f"expected (|beta-TRUE|)    = {exp:.8f}")
print(f"max deviation from expected = {np.max(np.abs(err.values-exp)):.2e}")
(C.ok if np.allclose(err.values,exp,atol=1e-9) else C.fail)(
    "counterfactual error is EXACTLY the estimation error -- construction adds no distortion")
assert np.allclose(err.values,exp,atol=1e-9)
C.ok("V05 COMPLETE")

# ---- MEMORY RELEASE (v14): free this model's frames so the next one starts clean ----
try:
    C.release(*[v for k,v in list(globals().items()) if k in ("panel","df","result") and v is not None])
except Exception as _e:
    print(f"[INFO]    release skipped: {_e}")
C.done(2, next_task="02_Core_DiD_Models/M01")